In [ ]:
# !pip install ipykernel
# !pip install transformers datasets matplotlib scikit-learn beautifulsoup4 trl pandas torch==2.8 peft liger-kernel
# !pip install flash-attn --no-build-isolation 

In [ ]:
# !pip install trl liger-kernel

In [ ]:
import os
GPU_DEVICES = [0]

os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'true'
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join([str(i) for i in GPU_DEVICES])

In [ ]:
import re
import random

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

from bs4 import BeautifulSoup, Comment
from datasets import load_dataset, concatenate_datasets, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    set_seed,
)
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

    # Makes CUDA deterministic (optional)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_all_seeds(42)

In [ ]:
N_SAMPLES = 10000

stream = load_dataset("phreshphish/phreshphish", split="train", streaming=True)
sampled = Dataset.from_list(list(stream.take(N_SAMPLES)))
print(f"Sampled {len(sampled)} examples")
print(sampled[0])

In [ ]:
print(f"=== Raw sampled dataset ===")
print(f"Total samples: {len(sampled)}")
print(f"Columns: {sampled.column_names}")
lang_counts = pd.Series(sampled['lang']).value_counts()
english_count = lang_counts.get('en', 0)
print(f"English: {english_count} ({english_count/len(sampled)*100:.1f}%) | Other languages: {len(sampled) - english_count}")
print(f"Labels: {pd.Series(sampled['label']).value_counts().to_string()}")
print()

dataset = sampled.filter(lambda x: x["lang"] == "en")
non_english_count = len(sampled) - len(dataset)
print(f"=== After language filter ===")
print(f"Kept: {len(dataset)} English | Removed: {non_english_count} non-English ({non_english_count/len(sampled)*100:.1f}%)")
print(f"Labels: {pd.Series(dataset['label']).value_counts().to_string()}")

In [ ]:
drop_cols = ["sha256", "target", "date", "lang", "lang_score"]
dataset = dataset.remove_columns(drop_cols)
print(f"Removed columns: {drop_cols}")
print(f"Remaining columns: {dataset.column_names}")
print(f"Total samples: {len(dataset)}")

In [ ]:
def plot_html_length_distribution(ds):
    html_lengths = [len(x) for x in ds["html"]]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(html_lengths, bins=50, edgecolor="black", alpha=0.7)
    ax.set_xlabel("HTML length (chars)")
    ax.set_ylabel("Count")
    ax.set_title("Distribution of HTML lengths")
    ax.axvline(np.median(html_lengths), color="red", linestyle="--", label=f"Median: {np.median(html_lengths):,.0f}")
    ax.axvline(np.mean(html_lengths), color="orange", linestyle="--", label=f"Mean: {np.mean(html_lengths):,.0f}")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"Min: {min(html_lengths):,} | Max: {max(html_lengths):,} | Median: {np.median(html_lengths):,.0f} | Mean: {np.mean(html_lengths):,.0f}")


plot_html_length_distribution(dataset)

In [ ]:
from lxml import html as lxml_html, etree

KEEP_TAGS = {"title", "meta", "form", "input", "button", "select", "textarea", "a", "img", "iframe", "label", "h1", "h2", "h3"}
KEEP_ATTRS = frozenset({"href", "src", "action", "method", "type", "name", "placeholder", "alt", "content", "value", "target"})

_REMOVE_TAGS = {"script", "style", "noscript"}

def clean_html(raw_html):
    try:
        doc = lxml_html.fromstring(raw_html)
    except Exception:
        return ""

    for el in list(doc.iter()):
        tag = el.tag
        if not isinstance(tag, str):
            p = el.getparent()
            if p is not None:
                p.remove(el)
        elif tag in _REMOVE_TAGS:
            p = el.getparent()
            if p is not None:
                p.remove(el)

    parts = []
    for el in doc.iter():
        if not isinstance(el.tag, str):
            continue
        if el.tag in KEEP_TAGS:
            kept = {k: v for k, v in el.items() if k in KEEP_ATTRS}
            tag = el.tag
            attr_str = " ".join(f'{k}="{v}"' for k, v in kept.items())
            open_tag = f"<{tag} {attr_str}>" if attr_str else f"<{tag}>"
            inner = el.text_content()
            parts.append(f"{open_tag}{inner}</{tag}>")
    return "\n".join(parts)

original_lengths = [len(x) for x in dataset["html"]]
dataset = dataset.map(
    lambda batch: {"html": [clean_html(h) for h in batch["html"]]},
    batched=True, batch_size=256,
)
cleaned_lengths = [len(x) for x in dataset["html"]]

print(f"=== HTML Cleaning Results ===")
print(f"Avg length before: {np.mean(original_lengths):,.0f} chars")
print(f"Avg length after:  {np.mean(cleaned_lengths):,.0f} chars")
print(f"Avg reduction:     {(1 - np.mean(cleaned_lengths)/np.mean(original_lengths))*100:.1f}%")

In [ ]:
plot_html_length_distribution(dataset)

In [ ]:
MAX_HTML_LEN = 20_000

before_count = len(dataset)
dataset = dataset.filter(lambda x: len(x["html"]) <= MAX_HTML_LEN)
print(f"=== HTML length filter ===")
print(f"Filtered HTML > {MAX_HTML_LEN:,} chars: {before_count} → {len(dataset)} samples (removed {before_count - len(dataset)})")
print(f"Labels: {pd.Series(dataset['label']).value_counts().to_string()}")
print()

benign = dataset.filter(lambda x: x["label"] == "benign")
phish = dataset.filter(lambda x: x["label"] == "phish")
min_count = min(len(benign), len(phish))

print(f"=== Balancing ===")
print(f"Before: benign={len(benign)}, phish={len(phish)}")
print(f"Downsampling to {min_count} per class")

benign = benign.shuffle(seed=42).select(range(min_count))
phish = phish.shuffle(seed=42).select(range(min_count))
balanced = concatenate_datasets([benign, phish]).shuffle(seed=42)

print()
print(f"=== Final balanced dataset ===")
print(f"Total: {len(balanced)} samples")
print(f"Labels: {pd.Series(balanced['label']).value_counts().to_string()}")

In [ ]:
train_test = balanced.train_test_split(test_size=0.2, seed=42)
train_ds = train_test["train"]
test_val = train_test["test"].train_test_split(test_size=0.5, seed=42)
val_ds = test_val["train"]
test_ds = test_val["test"]

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
for name, ds in [("Train", train_ds), ("Val", val_ds), ("Test", test_ds)]:
    counts = pd.Series(ds["label"]).value_counts()
    print(f"  {name}: benign={counts.get('benign', 0)}, phish={counts.get('phish', 0)}")

In [ ]:
# Load Tokenizer and Model
model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, attn_implementation="sdpa", torch_dtype=torch.bfloat16,
).to("cuda")
model = torch.compile(model)

In [ ]:
prompt_template = """Analyze the following webpage and determine if it is a phishing page or a benign (legitimate) page.

URL: {url}

HTML content:
{html}

Classify this webpage as either "phish" or "benign". Output your prediction in XML format: <prediction>phish</prediction> or <prediction>benign</prediction>"""

def preprocess_function(examples):
    urls = examples["url"]
    htmls = examples["html"]
    labels = examples["label"]

    prompts = [prompt_template.format(url=url, html=html) for url, html in zip(urls, htmls)]

    messages = [
        [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": f"<prediction>{label}</prediction>"},
        ]
        for prompt, label in zip(prompts, labels)
    ]

    return {"messages": messages}


In [ ]:
train_ds = train_ds.map(preprocess_function, batched=True)
val_ds = val_ds.map(preprocess_function, batched=True)
test_ds = test_ds.map(preprocess_function, batched=True)

print("Sample message:")
for msg in train_ds[0]["messages"]:
    content_preview = msg["content"][:300] + "..." if len(msg["content"]) > 300 else msg["content"]
    print(f"  [{msg['role']}]: {content_preview}")

In [ ]:
train_ds[0]

In [ ]:
def evaluate_model(model, tokenizer, eval_ds, batch_size=32, max_new_tokens=128):
    model.eval()
    responses = []
    labels = []
    tokenizer.padding_side = "left"

    with torch.inference_mode():
        for start_idx in tqdm(range(0, len(eval_ds), batch_size)):
            end_idx = min(start_idx + batch_size, len(eval_ds))

            batch_messages = [eval_ds[i]["messages"][:1] for i in range(start_idx, end_idx)]
            batch_labels = [eval_ds[i]["label"] for i in range(start_idx, end_idx)]

            texts = [
                tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
                for messages in batch_messages
            ]

            model_inputs = tokenizer(
                texts,
                return_tensors="pt",
                padding=True,
            ).to(model.device)

            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

            generated_ids = [
                output_ids[len(input_ids):]
                for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
            ]

            batch_responses = tokenizer.batch_decode(
                generated_ids,
                skip_special_tokens=True,
            )

            responses.extend(batch_responses)
            labels.extend(batch_labels)

    def parse_prediction(response):
        match = re.search(r"<prediction>(.*?)</prediction>", response)
        return match.group(1).strip().lower() if match else response.strip().lower()

    parsed = [parse_prediction(r) for r in responses]
    acc = accuracy_score(labels, parsed)
    valid_labels = [l for l, p in zip(labels, parsed) if p in ("phish", "benign")]
    valid_responses = [p for p in parsed if p in ("phish", "benign")]
    invalid_count = len(parsed) - len(valid_responses)

    print(f"Accuracy:  {100 * acc:.2f}%")
    if valid_responses:
        print(f"Precision: {100 * precision_score(valid_labels, valid_responses, pos_label='phish'):.2f}%")
        print(f"Recall:    {100 * recall_score(valid_labels, valid_responses, pos_label='phish'):.2f}%")
        print(f"F1:        {100 * f1_score(valid_labels, valid_responses, pos_label='phish'):.2f}%")
        print(f"\nConfusion matrix (rows=true, cols=pred):")
        cm = confusion_matrix(valid_labels, valid_responses, labels=["benign", "phish"])
        print(pd.DataFrame(cm, index=["true_benign", "true_phish"], columns=["pred_benign", "pred_phish"]))
    if invalid_count > 0:
        print(f"\nInvalid responses: {invalid_count}/{len(responses)}")
        print(f"Examples: {[r for r, p in zip(responses, parsed) if p not in ('phish', 'benign')][:5]}")

    return {
        "responses": responses,
        "parsed": parsed,
        "labels": labels,
        "accuracy": acc,
    }


In [ ]:
results = evaluate_model(model, tokenizer, val_ds)

In [ ]:
extra_kwargs = {
    "optim": "adamw_torch_fused",
    "learning_rate": 3e-04,
    "lr_scheduler_type": "constant_with_warmup",
    "max_grad_norm": 0.2,
    "warmup_steps": 5,
    "weight_decay": 0.1,

    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "max_steps": 100
}

peft_config = None # or peft_config = LoraConfig(???)


training_args = SFTConfig(
    output_dir="./results",    
    eval_strategy="steps",
    eval_steps=25,                  
    logging_strategy="steps",
    logging_steps=25,                   
    save_strategy="no",
    logging_dir="./logs",
    max_length=4096,
    packing=True,
    seed=42,
    data_seed=42,
    report_to="none",
    **extra_kwargs
)

In [ ]:
# Trainer Setup
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=peft_config,
)
trainer.can_return_loss = True

In [ ]:
trainer.train()

In [ ]:
results = evaluate_model(model, tokenizer, val_ds)

In [ ]:
results = evaluate_model(model, tokenizer, test_ds)